# Phase 2 conservative_anchored_baseline_001 — no-submit

Runs λ=1e-5 and λ=3e-5 only. No Kaggle submission commands are present.

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


In [ ]:
from pathlib import Path
Path('/kaggle/working/scripts').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/artifacts/conservative_anchored_baseline_001').mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/scripts/reproduce_conservative_anchored_baseline.py
#!/usr/bin/env python3
"""Phase 2 no-submit conservative anchored baseline.

Protocol:
- Detectron2 DefaultTrainer on unlearn20 with empty annotations
- poisoned_model.pth initialization
- official RetinaNet config and anchors
- LR=1e-4, batch size=4, MAX_ITER=20, STEPS=[]
- full model trainable (no explicit freeze)
- L2-SP style conservative anchor loss toward poisoned initial checkpoint
- correct 16-bit preprocessing
- DefaultPredictor inference at CONF_THRESH=0.2
- writes local CSV only; never submits
"""
from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import math
import os
import time
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import cv2
import numpy as np
import torch

torch.backends.nnpack.enabled = False
torch.set_num_threads(int(os.environ.get("TORCH_NUM_THREADS", "1")))
torch.set_num_interop_threads(int(os.environ.get("TORCH_NUM_INTEROP_THREADS", "1")))

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, DatasetMapper, MetadataCatalog, build_detection_train_loader, detection_utils as utils
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.engine.train_loop import SimpleTrainer
from tqdm import tqdm

ROOT = Path(__file__).resolve().parents[1]
DEFAULT_CKPT = ROOT / "data/raw/poisoned_model.pth"
DEFAULT_UNLEARN_DIR = ROOT / "data/raw"
DEFAULT_TEST_DIR = ROOT / "data/test_download/extracted/test_set/test_set"
DEFAULT_SAMPLE = ROOT / "data/test_download/extracted/sample_submission.csv"
DEFAULT_OUTDIR = ROOT / "artifacts/conservative_anchored_baseline_001"
EXPECTED_CKPT_SHA256 = "f6c21faa2a5b56549fc9e058147c90b149a034858fe0678f5a99ea5a6f0e657c"

BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
UNLEARN_LR = 1e-4
UNLEARN_ITERS = 20
BATCH_SIZE = 4
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024
DATASET_NAME = "neural_debris_unlearn_empty_strict"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read_sample(path: Path) -> List[Dict[str, str]]:
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows or list(rows[0].keys()) != ["id", "image_id", "prediction_string"]:
        raise ValueError(f"bad sample schema: {list(rows[0].keys()) if rows else None}")
    return rows


def load_16bit_scaled(path: Path) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


def register_unlearn_empty(unlearn_dir: Path) -> List[Dict[str, object]]:
    if DATASET_NAME in DatasetCatalog.list():
        DatasetCatalog.remove(DATASET_NAME)
        MetadataCatalog.remove(DATASET_NAME)
    json_path = unlearn_dir / "annotations_coco.json"
    with open(json_path, encoding="utf-8") as f:
        coco = json.load(f)
    dicts: List[Dict[str, object]] = []
    for im in coco["images"]:
        file_name = unlearn_dir / im["file_name"]
        if not file_name.exists():
            raise FileNotFoundError(file_name)
        dicts.append({
            "file_name": str(file_name),
            "height": im["height"],
            "width": im["width"],
            "image_id": im["id"],
            "annotations": [],
        })
    DatasetCatalog.register(DATASET_NAME, lambda dicts=dicts: dicts)
    MetadataCatalog.get(DATASET_NAME).set(thing_classes=["object"])
    return dicts


class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_16bit_scaled(Path(dataset_dict["file_name"]))
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


class UnlearnTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)


class AnchoredTrainer(UnlearnTrainer):
    """Minimal DefaultTrainer subclass adding an L2-SP anchor loss.

    It preserves the official DefaultTrainer construction, dataloader, optimizer,
    scheduler, checkpointer, and training constants. Only run_step is overridden
    to add loss_anchor to the Detectron2 loss dict.
    """

    def setup_anchor(self, anchor_lambda: float):
        self.anchor_lambda = float(anchor_lambda)
        self.anchor_params = {
            name: p.detach().clone()
            for name, p in self.model.named_parameters()
            if p.requires_grad
        }
        self.anchor_group_weights = {}
        for name in self.anchor_params:
            self.anchor_group_weights[name] = self._anchor_group_weight(name)

    @staticmethod
    def _anchor_group(name: str) -> str:
        lname = name.lower()
        if lname.endswith('.bias') or '.bias' in lname:
            return 'bias'
        if any(k in lname for k in ['norm', 'bn', 'gn']):
            return 'normalization'
        if 'backbone' in lname:
            if 'fpn' in lname or 'top_block' in lname:
                return 'fpn'
            return 'backbone'
        if 'cls' in lname or 'classification' in lname:
            return 'retinanet_cls_head'
        if 'bbox' in lname or 'box' in lname:
            return 'retinanet_bbox_head'
        if 'head' in lname:
            return 'retinanet_other_head'
        return 'other'

    @classmethod
    def _anchor_group_weight(cls, name: str) -> float:
        group = cls._anchor_group(name)
        if group in ['backbone', 'fpn']:
            return 1.0
        if group in ['retinanet_cls_head', 'retinanet_bbox_head', 'retinanet_other_head']:
            return 0.25
        if group == 'normalization':
            return 0.1
        if group == 'bias':
            return 0.0
        return 0.25

    def anchor_loss(self) -> torch.Tensor:
        if not getattr(self, 'anchor_params', None) or self.anchor_lambda <= 0:
            return torch.zeros((), device=self.model.device if hasattr(self.model, 'device') else next(self.model.parameters()).device)
        weighted = []
        for name, p in self.model.named_parameters():
            if not p.requires_grad or name not in self.anchor_params:
                continue
            w = float(self.anchor_group_weights.get(name, 0.0))
            if w <= 0:
                continue
            anchor = self.anchor_params[name].to(device=p.device, dtype=p.dtype)
            weighted.append(((p - anchor) ** 2).mean() * w)
        if not weighted:
            return torch.zeros((), device=next(self.model.parameters()).device)
        return torch.stack(weighted).mean()

    def run_step(self):
        assert self.model.training, '[AnchoredTrainer] model was changed to eval mode!'
        start = time.perf_counter()
        data = next(self._trainer._data_loader_iter)
        data_time = time.perf_counter() - start
        loss_dict = self.model(data)
        loss_anchor_unscaled = self.anchor_loss()
        loss_dict['loss_anchor'] = loss_anchor_unscaled * float(self.anchor_lambda)
        losses = sum(loss_dict.values())
        self._trainer.optimizer.zero_grad()
        losses.backward()
        self._trainer.optimizer.step()
        SimpleTrainer.write_metrics(loss_dict, data_time, self.iter)

    def drift_audit(self) -> Dict[str, object]:
        groups: Dict[str, Dict[str, float]] = {}
        global_diff_sq = 0.0
        global_anchor_sq = 0.0
        global_count = 0
        for name, p in self.model.named_parameters():
            if name not in getattr(self, 'anchor_params', {}):
                continue
            group = self._anchor_group(name)
            anchor = self.anchor_params[name].to(device=p.device, dtype=p.dtype)
            diff = (p.detach() - anchor).float()
            anch = anchor.float()
            d2 = float((diff * diff).sum().item())
            a2 = float((anch * anch).sum().item())
            n = int(p.numel())
            g = groups.setdefault(group, {'diff_sq': 0.0, 'anchor_sq': 0.0, 'count': 0, 'tensors': 0, 'anchor_weight_sum': 0.0})
            g['diff_sq'] += d2; g['anchor_sq'] += a2; g['count'] += n; g['tensors'] += 1; g['anchor_weight_sum'] += float(self.anchor_group_weights.get(name, 0.0))
            global_diff_sq += d2; global_anchor_sq += a2; global_count += n
        for g in groups.values():
            g['l2'] = float(g['diff_sq'] ** 0.5)
            g['anchor_l2'] = float(g['anchor_sq'] ** 0.5)
            g['relative_l2'] = float((g['diff_sq'] ** 0.5) / max(g['anchor_sq'] ** 0.5, 1e-12))
            g['rmse'] = float((g['diff_sq'] / max(g['count'], 1)) ** 0.5)
        return {
            'global': {
                'diff_sq': global_diff_sq,
                'anchor_sq': global_anchor_sq,
                'count': global_count,
                'l2': float(global_diff_sq ** 0.5),
                'anchor_l2': float(global_anchor_sq ** 0.5),
                'relative_l2': float((global_diff_sq ** 0.5) / max(global_anchor_sq ** 0.5, 1e-12)),
                'rmse': float((global_diff_sq / max(global_count, 1)) ** 0.5),
            },
            'groups': groups,
        }


def make_cfg(weights: Path, outdir: Path, *, train: bool, device: str):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = str(weights)
    cfg.MODEL.DEVICE = device
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
    cfg.DATASETS.TRAIN = (DATASET_NAME,) if train else ()
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = int(os.environ.get("D2_NUM_WORKERS", "2")) if train else 0
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = UNLEARN_LR
    cfg.SOLVER.MAX_ITER = UNLEARN_ITERS
    cfg.SOLVER.STEPS = []
    cfg.OUTPUT_DIR = str(outdir)
    return cfg


def count_trainable_params(model) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": int(total), "trainable_params": int(trainable), "frozen_params": int(total - trainable)}


def train_baseline(args) -> Path:
    t0 = time.perf_counter()
    args.outdir.mkdir(parents=True, exist_ok=True)
    ckpt_sha = sha256_file(args.checkpoint)
    if ckpt_sha != EXPECTED_CKPT_SHA256:
        raise ValueError(f"checkpoint sha mismatch: {ckpt_sha}")
    dicts = register_unlearn_empty(args.unlearn_dir)
    cfg = make_cfg(args.checkpoint, args.outdir, train=True, device=args.device)
    trainer = AnchoredTrainer(cfg)
    param_counts = count_trainable_params(trainer.model)
    trainer.resume_or_load(resume=False)
    trainer.setup_anchor(anchor_lambda=float(args.anchor_lambda))
    pretrain_anchor_loss = float(trainer.anchor_loss().detach().cpu().item())
    trainer.train()
    posttrain_anchor_loss = float(trainer.anchor_loss().detach().cpu().item())
    drift = trainer.drift_audit()
    final_ckpt = args.outdir / "model_final.pth"
    if not final_ckpt.exists():
        raise FileNotFoundError(final_ckpt)
    audit = {
        "mode": "conservative_anchored_baseline_001_train_no_submit",
        "variant": args.variant_label,
        "submitted": False,
        "auto_submit": False,
        "submission_authorized": False,
        "paid_gpu_authorized": False,
        "source_checkpoint": str(args.checkpoint),
        "source_checkpoint_sha256": ckpt_sha,
        "output_checkpoint": str(final_ckpt),
        "output_checkpoint_sha256": sha256_file(final_ckpt),
        "dataset": DATASET_NAME,
        "unlearn_dir": str(args.unlearn_dir),
        "unlearn_images": len(dicts),
        "annotations_empty": True,
        "config": {
            "BASE_CONFIG": BASE_CONFIG,
            "ANCHOR_ASPECT_RATIOS": ANCHOR_ASPECT_RATIOS,
            "ANCHOR_SIZES": ANCHOR_SIZES,
            "NUM_CLASSES": NUM_CLASSES,
            "UNLEARN_LR": UNLEARN_LR,
            "UNLEARN_ITERS": UNLEARN_ITERS,
            "MAX_ITER": int(cfg.SOLVER.MAX_ITER),
            "BATCH_SIZE": BATCH_SIZE,
            "CONF_THRESH": CONF_THRESH,
            "SOLVER_STEPS": list(cfg.SOLVER.STEPS),
            "SOLVER_OPTIMIZER": "Detectron2 default SGD",
            "MODEL_DEVICE": cfg.MODEL.DEVICE,
            "DATALOADER_NUM_WORKERS": cfg.DATALOADER.NUM_WORKERS,
        },
        "anchoring": {
            "type": "L2-SP_to_initial_poisoned_checkpoint",
            "lambda": float(args.anchor_lambda),
            "loss_formula": "loss_total = detectron2_empty_label_loss + lambda * mean_group_weighted_mean_squared_param_drift",
            "group_weights": {
                "backbone": 1.0,
                "fpn": 1.0,
                "retinanet_cls_head": 0.25,
                "retinanet_bbox_head": 0.25,
                "retinanet_other_head": 0.25,
                "normalization": 0.1,
                "bias": 0.0,
                "other": 0.25,
            },
            "pretrain_anchor_loss_unscaled": pretrain_anchor_loss,
            "posttrain_anchor_loss_unscaled": posttrain_anchor_loss,
            "posttrain_anchor_loss_scaled": posttrain_anchor_loss * float(args.anchor_lambda),
            "drift_audit_vs_initial": drift,
        },
        "trainable_scope": "full_detectron2_model_no_explicit_freeze_with_group_weighted_anchor_loss",
        "param_counts_after_build": param_counts,
        "runtime_sec": time.perf_counter() - t0,
    }
    (args.outdir / "strict_train_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(json.dumps(audit, indent=2), flush=True)
    return final_ckpt

def pred_string_from_instances(instances) -> str:
    out = instances.to("cpu")
    boxes = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    parts: List[str] = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])
    return " ".join(parts) or " "


def valid_pred_string(s: str) -> Dict[str, object]:
    vals = str(s).strip().split()
    if not vals:
        return {"valid": True, "num_detections": 0, "finite": True}
    ok_group = len(vals) % 5 == 0
    finite = True
    for v in vals:
        try:
            if not math.isfinite(float(v)):
                finite = False
        except Exception:
            finite = False
    return {"valid": ok_group and finite, "num_detections": len(vals) // 5 if ok_group else None, "finite": finite}


def infer_csv(args, checkpoint: Path) -> Path:
    t0 = time.perf_counter()
    args.outdir.mkdir(parents=True, exist_ok=True)
    rows_all = read_sample(args.sample_submission)
    rows = rows_all[args.start_index:]
    if args.max_images:
        rows = rows[: args.max_images]
    if not rows:
        raise ValueError("no rows selected")
    missing = [r["image_id"] for r in rows if not (args.test_dir / f"{r['image_id']}.png").exists()]
    if missing:
        raise FileNotFoundError(f"missing test images count={len(missing)} first={missing[:5]}")
    cfg = make_cfg(checkpoint, args.outdir, train=False, device=args.device)
    cfg.MODEL.WEIGHTS = str(checkpoint)
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
    predictor = DefaultPredictor(cfg)
    suffix = "" if args.start_index == 0 and not args.max_images else f"_part_start{args.start_index}_n{len(rows)}"
    out_csv = args.outdir / f"strict_simple_finetuning_baseline_submission{suffix}.csv"
    audit_path = args.outdir / f"strict_simple_finetuning_baseline_submission{suffix}_audit.json"

    total_det = empty = invalid = nonfinite = 0
    scores_all: List[float] = []
    widths: List[float] = []
    heights: List[float] = []
    top_scores: List[float] = []
    first20 = []
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "image_id", "prediction_string"])
        writer.writeheader()
        for idx, r in enumerate(tqdm(rows, desc=f"Strict baseline inference {args.start_index}+{len(rows)}"), start=1):
            image_id = r["image_id"]
            im = load_16bit_scaled(args.test_dir / f"{image_id}.png")
            out = predictor(im)["instances"].to("cpu")
            pred_s = pred_string_from_instances(out)
            chk = valid_pred_string(pred_s)
            vals = pred_s.strip().split()
            scores = []
            if vals:
                nums = [float(x) for x in vals]
                scores = nums[0::5]
                scores_all.extend(scores)
                widths.extend(nums[3::5])
                heights.extend(nums[4::5])
                top_scores.append(max(scores))
            nd = int(chk["num_detections"] or 0)
            total_det += nd
            empty += int(nd == 0)
            invalid += int(not chk["valid"])
            nonfinite += int(not chk["finite"])
            writer.writerow({"id": r["id"], "image_id": r["image_id"], "prediction_string": pred_s})
            if len(first20) < 20:
                first20.append({"id": r["id"], "image_id": r["image_id"], "num_detections": nd, "top_score": max(scores) if scores else None})
            if idx % 50 == 0:
                f.flush()

    def pct(vals: List[float], q: float):
        if not vals:
            return None
        return float(np.quantile(np.asarray(vals, dtype=np.float64), q))

    audit = {
        "mode": "conservative_anchored_baseline_001_inference_no_submit",
        "variant": args.variant_label,
        "submitted": False,
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
        "sample_submission": str(args.sample_submission),
        "test_dir": str(args.test_dir),
        "output_csv": str(out_csv),
        "output_csv_sha256": sha256_file(out_csv),
        "config": {
            "BASE_CONFIG": BASE_CONFIG,
            "ANCHOR_ASPECT_RATIOS": ANCHOR_ASPECT_RATIOS,
            "ANCHOR_SIZES": ANCHOR_SIZES,
            "NUM_CLASSES": NUM_CLASSES,
            "CONF_THRESH": CONF_THRESH,
            "device": args.device,
            "TEST_DETECTIONS_PER_IMAGE": int(cfg.TEST.DETECTIONS_PER_IMAGE),
            "RETINANET_TOPK_CANDIDATES_TEST": int(cfg.MODEL.RETINANET.TOPK_CANDIDATES_TEST),
            "RETINANET_NMS_THRESH_TEST": float(cfg.MODEL.RETINANET.NMS_THRESH_TEST),
        },
        "row_count_total_sample": len(rows_all),
        "row_start_index": args.start_index,
        "row_count": len(rows),
        "detections_total": total_det,
        "detections_per_image_mean": total_det / len(rows),
        "empty_prediction_images": empty,
        "invalid_prediction_strings": invalid,
        "nonfinite_values": nonfinite,
        "score_distribution": {"count": len(scores_all), "min": pct(scores_all, 0), "p25": pct(scores_all, 0.25), "median": pct(scores_all, 0.5), "p75": pct(scores_all, 0.75), "p95": pct(scores_all, 0.95), "max": pct(scores_all, 1)},
        "top_score_distribution": {"count": len(top_scores), "median": pct(top_scores, 0.5), "p25": pct(top_scores, 0.25), "p75": pct(top_scores, 0.75), "min": pct(top_scores, 0), "max": pct(top_scores, 1)},
        "box_width_distribution": {"count": len(widths), "median": pct(widths, 0.5), "p25": pct(widths, 0.25), "p75": pct(widths, 0.75), "min": pct(widths, 0), "max": pct(widths, 1)},
        "box_height_distribution": {"count": len(heights), "median": pct(heights, 0.5), "p25": pct(heights, 0.25), "p75": pct(heights, 0.75), "min": pct(heights, 0), "max": pct(heights, 1)},
        "per_image_summary_first20": first20,
        "runtime_sec": time.perf_counter() - t0,
    }
    audit_path.write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
    print(json.dumps({"csv": str(out_csv), "audit": str(audit_path), "rows": len(rows), "detections_total": total_det, "empty": empty, "invalid": invalid, "nonfinite": nonfinite, "runtime_sec": audit["runtime_sec"]}, indent=2), flush=True)
    return out_csv


def main(argv: Optional[Iterable[str]] = None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", type=Path, default=DEFAULT_CKPT)
    ap.add_argument("--unlearn-dir", type=Path, default=DEFAULT_UNLEARN_DIR)
    ap.add_argument("--test-dir", type=Path, default=DEFAULT_TEST_DIR)
    ap.add_argument("--sample-submission", type=Path, default=DEFAULT_SAMPLE)
    ap.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR)
    ap.add_argument("--device", choices=["cpu", "cuda"], default="cpu")
    ap.add_argument("--train-only", action="store_true")
    ap.add_argument("--infer-only", action="store_true")
    ap.add_argument("--inference-checkpoint", type=Path, default=None)
    ap.add_argument("--anchor-lambda", type=float, required=True)
    ap.add_argument("--variant-label", type=str, required=True)
    ap.add_argument("--start-index", type=int, default=0)
    ap.add_argument("--max-images", type=int, default=0)
    args = ap.parse_args(list(argv) if argv is not None else None)
    if args.train_only and args.infer_only:
        raise ValueError("choose at most one of --train-only/--infer-only")
    ckpt = args.inference_checkpoint
    if not args.infer_only:
        ckpt = train_baseline(args)
    if not args.train_only:
        if ckpt is None:
            ckpt = args.outdir / "model_final.pth"
        infer_csv(args, ckpt)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Phase 2 conservative anchored baseline: strict no-submit, only lambda changes.
import os, subprocess, sys, json, time
from pathlib import Path

os.environ.setdefault("TORCH_NUM_THREADS", "2")
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("D2_NUM_WORKERS", "2")

ROOT = Path("/kaggle/working")
SCRIPT = ROOT / "scripts/reproduce_conservative_anchored_baseline.py"
BASE_OUT = ROOT / "artifacts/conservative_anchored_baseline_001"
BASE_OUT.mkdir(parents=True, exist_ok=True)

CHECKPOINT = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth")
UNLEARN_DIR = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set")
TEST_DIR = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set")
SAMPLE = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/sample_submission.csv")

variants = [
    {"label": "lambda_1e-5", "lambda": 1e-5},
    {"label": "lambda_3e-5", "lambda": 3e-5},
]
run_manifest = {
    "mode": "conservative_anchored_baseline_001_no_submit",
    "variants": variants,
    "reference_iter20_score": 259.7886,
    "strict_constants": {
        "MAX_ITER": 20,
        "UNLEARN_LR": 1e-4,
        "BATCH_SIZE": 4,
        "CONF_THRESH": 0.2,
        "anchors_unchanged": True,
        "preprocessing_16bit_unchanged": True,
        "DefaultTrainer_minimal_subclass": True,
        "DefaultPredictor": True,
        "labels_empty": True,
    },
    "submitted": False,
    "auto_submit": False,
    "submission_authorized": False,
    "paid_gpu_authorized": False,
    "started_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "results": [],
}

for v in variants:
    outdir = BASE_OUT / v["label"]
    outdir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, str(SCRIPT),
        "--checkpoint", str(CHECKPOINT),
        "--unlearn-dir", str(UNLEARN_DIR),
        "--test-dir", str(TEST_DIR),
        "--sample-submission", str(SAMPLE),
        "--outdir", str(outdir),
        "--device", "cuda",
        "--anchor-lambda", str(v["lambda"]),
        "--variant-label", v["label"],
    ]
    t0 = time.time()
    print("RUN", json.dumps({"variant": v, "cmd": cmd}, indent=2), flush=True)
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (outdir / "run_stdout.log").write_text(proc.stdout, encoding="utf-8")
    result = {"variant": v, "returncode": proc.returncode, "runtime_sec_wall": time.time()-t0, "outdir": str(outdir)}
    run_manifest["results"].append(result)
    print(proc.stdout[-4000:], flush=True)
    if proc.returncode != 0:
        run_manifest["failed_at_variant"] = v
        (BASE_OUT / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
        raise SystemExit(proc.returncode)

run_manifest["finished_at_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
(BASE_OUT / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print(json.dumps(run_manifest, indent=2), flush=True)


In [ ]:
# Build SHA256 manifest for all outputs; still no-submit.
from pathlib import Path
import hashlib, json, time
BASE_OUT = Path("/kaggle/working/artifacts/conservative_anchored_baseline_001")
paths = [Path("/kaggle/working/scripts/reproduce_conservative_anchored_baseline.py")]
paths.extend([p for p in BASE_OUT.rglob("*") if p.is_file()])
records=[]
for p in sorted(paths):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""):
            h.update(chunk)
    records.append({"path": str(p), "relpath": str(p.relative_to(Path('/kaggle/working'))), "size": p.stat().st_size, "sha256": h.hexdigest()})
manifest={"created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "mode":"conservative_anchored_baseline_001_no_submit", "submitted": False, "files": records}
(BASE_OUT / "sha256_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({"manifest": str(BASE_OUT/'sha256_manifest.json'), "file_count": len(records)}, indent=2))


Expected outputs under `/kaggle/working/artifacts/conservative_anchored_baseline_001/lambda_1e-5` and `lambda_3e-5`: checkpoint, CSV, train/inference audits, logs, and global `sha256_manifest.json`.